In [216]:
# 데이터 생성
import pandas as pd
dataset = pd.read_csv('../../data/csv/movie_reviews3.csv')

In [217]:
# 독립, 종속
X = dataset.iloc[ :, -1]
# 0.5 => 0 으로, 3 => 1로, 5 => 2로
def convert(score):
  return 0 if score == 0.5 else 1 if score == 3 else 2
y = dataset['label'].apply(convert)

In [218]:
# 형태소 분석
from konlpy.tag import Okt
okt = Okt()

def tokenized_korean(text_list):
  return ["".join(okt.morphs(text, stem=True)) for text in text_list]

morphs_korean = tokenized_korean(X)

In [219]:
# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode='int',
	output_sequence_length = 10
)

vectorize_layer.adapt(morphs_korean)

In [220]:
# 파이프라인
import tensorflow as tf
train_ds = tf.data.Dataset.from_tensor_slices((morphs_korean,y)).batch(16)


In [221]:
# 모델 설계
def build_positional_encoding():
  
  # 입력층
  inputs = layers.Input((1,), dtype=tf.string)
  
  # 벡터화
  x = vectorize_layer(inputs)
  
  # 임베딩 : 단어를
  vocab_size = vectorize_layer.vocabulary_size()
  word_embeding = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)
  
  # 포지셔널 인코딩
  # 위치 정보 추가 : 0~9 번호를 생성
  positions = tf.range(start=0, limit=10, delta=1)
  # 임베딩 : 위치를
  pos_embeding = layers.Embedding(input_dim=10, output_dim=64)(positions)
  
  # 의미 + 위치
  x = word_embeding + pos_embeding
  
  # 멀티헤드 어텐션
  attention_output = layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)
  
  # 잔차 계산 및 정규화
  x = layers.Add()([x,attention_output])
  x = layers.LayerNormalization()(x)
  
  # ffn
  ffn = layers.Dense(64, activation='relu')(x)
  x = layers.Add()([x, ffn])
  x = layers.LayerNormalization()(x)
  
  # 압축
  x = layers.GlobalAveragePooling1D()(x)
  
  # 출력층
  outputs = layers.Dense(3, activation='softmax')(x)
  
  return models.Model(inputs=inputs, outputs=outputs)

model = build_positional_encoding()

In [222]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [223]:
# 모델 학습
model.fit(train_ds, epochs=50)

Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.3221 - loss: 1.3188
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4375 - loss: 1.1051
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3798 - loss: 1.0440
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6827 - loss: 0.9426
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6635 - loss: 0.8774
Epoch 6/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8173 - loss: 0.7899
Epoch 7/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9183 - loss: 0.6908
Epoch 8/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9952 - loss: 0.5854
Epoch 9/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.4314
Epoch 10/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1954
Epoch 11/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0486
Epoch 12/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - lo

In [224]:
import numpy as np
#예측

# 테스트
test_text = ["영화가 너무 재미있어서 하나도 안 지루해요", "영화가 하나도 안 재미있어서 너무 지루해요.",
             "기대 많이 했는데 생각보다 너무 별로네요.", "세련된 연출인 줄 알았는데 너무 촌스러워요."]

# 형태소별로 문장을 수정
test_morphs = tokenized_korean(test_text)

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)

# 매핑
labels = {0: 0.5, 1: 3, 2: 5}

for i in range(predictions_size):
    predicted_class = np.argmax(predictions[i])
    predicted_score = labels[predicted_class]
    print(f'{test_text[i]}: 예측 평점 = {predicted_score}점')
    
  

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step
영화가 너무 재미있어서 하나도 안 지루해요: 예측 평점 = 5점
영화가 하나도 안 재미있어서 너무 지루해요.: 예측 평점 = 5점
기대 많이 했는데 생각보다 너무 별로네요.: 예측 평점 = 3점
세련된 연출인 줄 알았는데 너무 촌스러워요.: 예측 평점 = 0.5점
